In [2]:
import numpy as np
import pandas as pd

import seaborn as sns
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.preprocessing import LabelEncoder, OneHotEncoder, PolynomialFeatures
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score, accuracy_score, confusion_matrix


import warnings
warnings.filterwarnings('ignore')

### Information to check before proceeding with the data cleaning

- InvoiceNo: Invoice number. Nominal. A 6-digit integral number uniquely assigned to each transaction. If this code starts with the letter 'c', it indicates a cancellation. 
- StockCode: Product (item) code. Nominal. A 5-digit integral number uniquely assigned to each distinct product. 
- Description: Product (item) name. Nominal. 
- Quantity: The quantities of each product (item) per transaction. Numeric.	
- InvoiceDate: Invice date and time. Numeric. The day and time when a transaction was generated. 
- UnitPrice: Unit price. Numeric. Product price per unit in sterling (Â£). 
- CustomerID: Customer number. Nominal. A 5-digit integral number uniquely assigned to each customer. 
- Country: Country name. Nominal. The name of the country where a customer resides.

In [3]:
df = pd.read_excel('online_retail_II.xlsx', sheet_name='Year 2009-2010')
df.head()

,Invoice,StockCode,Description,Quantity,InvoiceDate,Price,Customer ID,Country
0,489434,85048,15CM CHRISTMAS GLASS BALL 20 LIGHTS,12,2009-12-01 07:45:00,6.95,13085.0,United Kingdom
1,489434,79323P,PINK CHERRY LIGHTS,12,2009-12-01 07:45:00,6.75,13085.0,United Kingdom
2,489434,79323W,WHITE CHERRY LIGHTS,12,2009-12-01 07:45:00,6.75,13085.0,United Kingdom
3,489434,22041,"RECORD FRAME 7"" SINGLE SIZE",48,2009-12-01 07:45:00,2.10,13085.0,United Kingdom
4,489434,21232,STRAWBERRY CERAMIC TRINKET BOX,24,2009-12-01 07:45:00,1.25,13085.0,United Kingdom


# **STEP 1 : DOING EDA**

In [4]:
# checking for nan values
print(df.isna().sum())
print("----------------------------------------")
print(df.isna().mean())         # have to delete all the nan rows

Invoice             0
StockCode           0
Description      2928
Quantity            0
InvoiceDate         0
Price               0
Customer ID    107927
Country             0
dtype: int64
----------------------------------------
Invoice        0.000000
StockCode      0.000000
Description    0.005572
Quantity       0.000000
InvoiceDate    0.000000
Price          0.000000
Customer ID    0.205395
Country        0.000000
dtype: float64


In [5]:
print(df.info())
print("-------------------------------")
df.describe()


<class 'pandas.core.frame.DataFrame'>
RangeIndex: 525461 entries, 0 to 525460
Data columns (total 8 columns):
 #   Column       Non-Null Count   Dtype         
---  ------       --------------   -----         
 0   Invoice      525461 non-null  object        
 1   StockCode    525461 non-null  object        
 2   Description  522533 non-null  object        
 3   Quantity     525461 non-null  int64         
 4   InvoiceDate  525461 non-null  datetime64[ns]
 5   Price        525461 non-null  float64       
 6   Customer ID  417534 non-null  float64       
 7   Country      525461 non-null  object        
dtypes: datetime64[ns](1), float64(2), int64(1), object(4)
memory usage: 32.1+ MB
None
-------------------------------


,Quantity,InvoiceDate,Price,Customer ID
count,525461.000000,525461,525461.000000,417534.000000
mean,10.337667,2010-06-28 11:37:36.845017856,4.688834,15360.645478
min,-9600.000000,2009-12-01 07:45:00,-53594.360000,12346.000000
25%,1.000000,2010-03-21 12:20:00,1.250000,13983.000000
50%,3.000000,2010-07-06 09:51:00,2.100000,15311.000000
75%,10.000000,2010-10-15 12:45:00,4.210000,16799.000000
max,19152.000000,2010-12-09 20:01:00,25111.090000,18287.000000
std,107.424110,NaN,146.126914,1680.811316


In [6]:
# as we can see some quantities and price are in negative values, we have to detect them first
print(df[(df['Price'] < 0)].shape)    # total of 3 rows have negative quantity
print(df[(df['Quantity'] < 0)].shape)    # total of 12326 rows have negative quantity
print(df[(df['Quantity'] < 0) | (df['Price'] < 0)].shape)    # in total we have to delete 12329 rows

df[df['Quantity'] < 0].head()

(3, 8)
(12326, 8)
(12329, 8)


,Invoice,StockCode,Description,Quantity,InvoiceDate,Price,Customer ID,Country
178,C489449,22087,PAPER BUNTING WHITE LACE,-12,2009-12-01 10:33:00,2.95,16321.0,Australia
179,C489449,85206A,CREAM FELT EASTER EGG BASKET,-6,2009-12-01 10:33:00,1.65,16321.0,Australia
180,C489449,21895,POTTING SHED SOW 'N' GROW SET,-4,2009-12-01 10:33:00,4.25,16321.0,Australia
181,C489449,21896,POTTING SHED TWINE,-6,2009-12-01 10:33:00,2.10,16321.0,Australia
182,C489449,22083,PAPER CHAIN KIT RETRO SPOT,-12,2009-12-01 10:33:00,2.95,16321.0,Australia


In [7]:
df.describe(include='O')    # printing the data which are object

,Invoice,StockCode,Description,Country
count,525461,525461,522533,525461
unique,28816,4632,4681,40
top,537434,85123A,WHITE HANGING HEART T-LIGHT HOLDER,United Kingdom
freq,675,3516,3549,485852


Other parameters you can use instead of 'O'

- `include='all'`
- `include='number'`
- `include='datetime'`
- `include=['O', 'number']`
- `include='category'`

In [8]:
print(df['Customer ID'].isna().shape)
# print(df['Invoice']isna().shape)
print(df['Invoice'].unique())       # invoice holds some numbers with 'C', which means cancellation

(525461,)
[489434 489435 489436 ... 538169 538170 538171]


**InvoiceNo: Invoice number. Nominal. A 6-digit integral number uniquely assigned to each transaction. If this code starts with the letter 'c', it indicates a cancellation.**

In [9]:
df['Invoice'] = df['Invoice'].astype(str)

# now we are going to use regex to find how many Invoice hodls texts other than integers
print(df[df['Invoice'].str.match('C')].shape)     # as you can see there are 10206 rows that holds cancellation data     
df[df['Invoice'].str.match('C')]     


(10206, 8)


,Invoice,StockCode,Description,Quantity,InvoiceDate,Price,Customer ID,Country
178,C489449,22087,PAPER BUNTING WHITE LACE,-12,2009-12-01 10:33:00,2.95,16321.0,Australia
179,C489449,85206A,CREAM FELT EASTER EGG BASKET,-6,2009-12-01 10:33:00,1.65,16321.0,Australia
180,C489449,21895,POTTING SHED SOW 'N' GROW SET,-4,2009-12-01 10:33:00,4.25,16321.0,Australia
181,C489449,21896,POTTING SHED TWINE,-6,2009-12-01 10:33:00,2.10,16321.0,Australia
182,C489449,22083,PAPER CHAIN KIT RETRO SPOT,-12,2009-12-01 10:33:00,2.95,16321.0,Australia
...,...,...,...,...,...,...,...,...
524695,C538123,22956,36 FOIL HEART CAKE CASES,-2,2010-12-09 15:41:00,2.10,12605.0,Germany
524696,C538124,M,Manual,-4,2010-12-09 15:43:00,0.50,15329.0,United Kingdom
524697,C538124,22699,ROSES REGENCY TEACUP AND SAUCER,-1,2010-12-09 15:43:00,2.95,15329.0,United Kingdom
524698,C538124,22423,REGENCY CAKESTAND 3 TIER,-1,2010-12-09 15:43:00,12.75,15329.0,United Kingdom


In [10]:
# printing the invoice numbers that doesn't start and end only with numbers
df[df['Invoice'].str.match("^\d+$") == False]



,Invoice,StockCode,Description,Quantity,InvoiceDate,Price,Customer ID,Country
178,C489449,22087,PAPER BUNTING WHITE LACE,-12,2009-12-01 10:33:00,2.95,16321.0,Australia
179,C489449,85206A,CREAM FELT EASTER EGG BASKET,-6,2009-12-01 10:33:00,1.65,16321.0,Australia
180,C489449,21895,POTTING SHED SOW 'N' GROW SET,-4,2009-12-01 10:33:00,4.25,16321.0,Australia
181,C489449,21896,POTTING SHED TWINE,-6,2009-12-01 10:33:00,2.10,16321.0,Australia
182,C489449,22083,PAPER CHAIN KIT RETRO SPOT,-12,2009-12-01 10:33:00,2.95,16321.0,Australia
...,...,...,...,...,...,...,...,...
524695,C538123,22956,36 FOIL HEART CAKE CASES,-2,2010-12-09 15:41:00,2.10,12605.0,Germany
524696,C538124,M,Manual,-4,2010-12-09 15:43:00,0.50,15329.0,United Kingdom
524697,C538124,22699,ROSES REGENCY TEACUP AND SAUCER,-1,2010-12-09 15:43:00,2.95,15329.0,United Kingdom
524698,C538124,22423,REGENCY CAKESTAND 3 TIER,-1,2010-12-09 15:43:00,12.75,15329.0,United Kingdom


In [11]:
#checking if there is any other letter apart from C
 
a = df['Invoice'].str.replace("[0-9]", "", regex=True)
print(a.unique())       # as you can see after replacing all the numbers with 'empty' we have another letter A other than letter C

['' 'C' 'A']


In [12]:
# a = df[df['Invoice'].str.contains('[CA]', na=False)].index
df[df['Invoice'].str.contains('[A]')]


,Invoice,StockCode,Description,Quantity,InvoiceDate,Price,Customer ID,Country
179403,A506401,B,Adjust bad debt,1,2010-04-29 13:36:00,-53594.36,NaN,United Kingdom
276274,A516228,B,Adjust bad debt,1,2010-07-19 11:24:00,-44031.79,NaN,United Kingdom
403472,A528059,B,Adjust bad debt,1,2010-10-20 12:04:00,-38925.87,NaN,United Kingdom


**StockCode: Product (item) code. Nominal. A 5-digit integral number uniquely assigned to each distinct product.**

In [13]:
df['StockCode'] = df['StockCode'].astype(str)

# printing the StockCode data which doesnot meet the requirement
df[df['StockCode'].str.contains("^\d{5}$") == False]

,Invoice,StockCode,Description,Quantity,InvoiceDate,Price,Customer ID,Country
1,489434,79323P,PINK CHERRY LIGHTS,12,2009-12-01 07:45:00,6.75,13085.0,United Kingdom
2,489434,79323W,WHITE CHERRY LIGHTS,12,2009-12-01 07:45:00,6.75,13085.0,United Kingdom
12,489436,48173C,DOOR MAT BLACK FLOCK,10,2009-12-01 09:06:00,5.95,13078.0,United Kingdom
23,489436,35004B,SET OF 3 BLACK FLYING DUCKS,12,2009-12-01 09:06:00,4.65,13078.0,United Kingdom
28,489436,84596F,SMALL MARSHMALLOWS PINK BOWL,8,2009-12-01 09:06:00,1.25,13078.0,United Kingdom
...,...,...,...,...,...,...,...,...
525387,538170,84029E,RED WOOLLY HOTTIE WHITE HEART.,2,2010-12-09 19:32:00,3.75,13969.0,United Kingdom
525388,538170,84029G,KNITTED UNION FLAG HOT WATER BOTTLE,2,2010-12-09 19:32:00,3.75,13969.0,United Kingdom
525389,538170,85232B,SET OF 3 BABUSHKA STACKING TINS,2,2010-12-09 19:32:00,4.95,13969.0,United Kingdom
525435,538171,47591D,PINK FAIRY CAKE CHILDRENS APRON,1,2010-12-09 20:01:00,1.95,17530.0,United Kingdom


In [14]:
df[df['StockCode'].str.contains("^\d{5}$") == False & df['StockCode'].str.contains("^\d{5}[a-zA-Z]$")]['StockCode'].shape
# df[df['StockCode'].str.contains("^\d{5}$") == False].shape        # you can also do this instead


(80112,)

In [15]:
# checking for all the StockCode with letters
checking_letters = df['StockCode'].str.replace("\d", "", regex=True)

unique = checking_letters.unique()
print(unique)
print()

# checking for all the StockCode data that contains a letter
for i in unique:
    if i == '':
        continue
    print(f"SHAPE OF '{i}' {df[df['StockCode'].str.contains(i)].shape}")

['' 'P' 'W' 'C' 'B' 'F' 'L' 'S' 'A' 'N' 'POST' 'E' 'J' 'D' 'G' 'LP' 'BL'
 'K' 'H' 'GR' 'M' 'DCGS' 'DOT' 'U' 'b' 'w' 'c' 'a' 'f' 'bl' 's' 'p' 'R'
 'V' 'T' 'I' 'BANK CHARGES' 'O' 'Z' 'TEST' 'gift__' 'DCGSN' 'm' 'PADS' 'Y'
 'HC' 'e' 'd' 'ADJUST' 'DCGSSGIRL' 'GIFT' 'DCGSLBOY' 'k' 'g' 'DCGSSBOY'
 'DCGSLGIRL' 'j' 'l' 'n' 'J ' 'SP' 'AMAZONFEE']

SHAPE OF 'P' (2401, 8)
SHAPE OF 'W' (1083, 8)
SHAPE OF 'C' (9657, 8)
SHAPE OF 'B' (22677, 8)
SHAPE OF 'F' (2712, 8)
SHAPE OF 'L' (3865, 8)
SHAPE OF 'S' (3716, 8)
SHAPE OF 'A' (18992, 8)
SHAPE OF 'N' (988, 8)
SHAPE OF 'POST' (865, 8)
SHAPE OF 'E' (3123, 8)
SHAPE OF 'J' (415, 8)
SHAPE OF 'D' (6746, 8)
SHAPE OF 'G' (2392, 8)
SHAPE OF 'LP' (231, 8)
SHAPE OF 'BL' (597, 8)
SHAPE OF 'K' (549, 8)
SHAPE OF 'H' (330, 8)
SHAPE OF 'GR' (122, 8)
SHAPE OF 'M' (1420, 8)
SHAPE OF 'DCGS' (113, 8)
SHAPE OF 'DOT' (736, 8)
SHAPE OF 'U' (297, 8)
SHAPE OF 'b' (352, 8)
SHAPE OF 'w' (1, 8)
SHAPE OF 'c' (360, 8)
SHAPE OF 'a' (409, 8)
SHAPE OF 'f' (90, 8)
SHAPE OF 'bl' (31, 8)

In [16]:
df[df['StockCode'].str.contains("PADS")]      # we will keep all the StockCode with PADS

,Invoice,StockCode,Description,Quantity,InvoiceDate,Price,Customer ID,Country
62299,494914,PADS,PADS TO MATCH ALL CUSHIONS,1,2010-01-19 17:04:00,0.001,16705.0,United Kingdom
74731,496222,PADS,PADS TO MATCH ALL CUSHIONS,1,2010-01-29 13:53:00,0.001,13583.0,United Kingdom
77702,496473,PADS,PADS TO MATCH ALL CUSHIONS,1,2010-02-01 15:38:00,0.001,17350.0,United Kingdom
79794,496643,PADS,PADS TO MATCH ALL CUSHIONS,1,2010-02-03 11:58:00,0.001,13408.0,United Kingdom
90798,497935,PADS,PADS TO MATCH ALL CUSHIONS,1,2010-02-15 10:47:00,0.001,13408.0,United Kingdom
97716,498562,PADS,PADS TO MATCH ALL CUSHIONS,1,2010-02-21 12:03:00,0.001,15182.0,United Kingdom
101718,499056,PADS,PADS TO MATCH ALL CUSHIONS,1,2010-02-24 13:46:00,0.001,13765.0,United Kingdom
104480,499399,PADS,PADS TO MATCH ALL CUSHIONS,1,2010-02-26 13:26:00,0.001,14459.0,United Kingdom
123947,501176,PADS,PADS TO MATCH ALL CUSHIONS,1,2010-03-15 11:00:00,0.001,14857.0,United Kingdom
156809,504332,PADS,PADS TO MATCH ALL CUSHIONS,1,2010-04-12 16:30:00,0.001,12671.0,Germany


# **STEP 2 : DATA CLEANING**

In [17]:
cleaned_df = df.copy()      # copying the data to prevent any changes into the original data

# cleaned_df[cleaned_df['StockCode'].str.contains("PADS")]


In [18]:
# making a mask for data cleaning
cleaned_df['Invoice'] = cleaned_df['Invoice'].astype("str")
cleaned_df['StockCode'] = cleaned_df['StockCode'].astype("str")


invoice_mask = cleaned_df['Invoice'].str.match(r"^\d{6}$")
stock_mask = (cleaned_df['StockCode'].str.match(r"^\d{5}+$") == True) | (cleaned_df['StockCode'].str.match(r"^\d{5}[a-zA-Z]+$")) | (cleaned_df['StockCode'].str.match(r"PADS") == True)



cleaned_df = cleaned_df[invoice_mask & stock_mask]
# cleaned_df[cleaned_df['Invoice'].str.match("^\d{6}$") == False]
cleaned_df.head()

,Invoice,StockCode,Description,Quantity,InvoiceDate,Price,Customer ID,Country
0,489434,85048,15CM CHRISTMAS GLASS BALL 20 LIGHTS,12,2009-12-01 07:45:00,6.95,13085.0,United Kingdom
1,489434,79323P,PINK CHERRY LIGHTS,12,2009-12-01 07:45:00,6.75,13085.0,United Kingdom
2,489434,79323W,WHITE CHERRY LIGHTS,12,2009-12-01 07:45:00,6.75,13085.0,United Kingdom
3,489434,22041,"RECORD FRAME 7"" SINGLE SIZE",48,2009-12-01 07:45:00,2.10,13085.0,United Kingdom
4,489434,21232,STRAWBERRY CERAMIC TRINKET BOX,24,2009-12-01 07:45:00,1.25,13085.0,United Kingdom


In [19]:
# dropping the NaN from Customer ID only
print(cleaned_df['Customer ID'].isna().sum())

# cleaned_df['Customer ID'].dropna()        # this will not work
cleaned_df.dropna(subset=['Customer ID'], inplace=True)   

print(cleaned_df['Customer ID'].isna().sum())


106459
0


In [20]:
# dropping the prices which are 0
print(len(cleaned_df[cleaned_df['Price'] == 0.0]))

cleaned_df = cleaned_df[cleaned_df['Price'] > 0]

print(len(cleaned_df[cleaned_df['Price'] == 0.0]))
print(cleaned_df['Price'].min())      # we still have a min price of 0.001


28
0
0.001


In [21]:
cleaned_df.describe()

,Quantity,InvoiceDate,Price,Customer ID
count,406309.000000,406309,406309.000000,406309.000000
mean,13.617924,2010-07-01 10:14:25.869572352,2.991668,15373.722268
min,1.000000,2009-12-01 07:45:00,0.001000,12346.000000
25%,2.000000,2010-03-26 14:01:00,1.250000,14006.000000
50%,5.000000,2010-07-09 15:48:00,1.950000,15326.000000
75%,12.000000,2010-10-14 17:09:00,3.750000,16814.000000
max,19152.000000,2010-12-09 20:01:00,295.000000,18287.000000
std,96.998833,NaN,4.285951,1677.329470


In [22]:
# now lets see how much data did we loose in the datacleaning process

print(np.round(len(cleaned_df) / len(df), 2))   # so we lost about 23 % of data


0.77


### 23 % of data lost during data cleaning

In [23]:
import pickle
cleaned_df.to_pickle("cleaned_online_retail.pkl")
print("FILE SAVED SUCCESSFULLY")

FILE SAVED SUCCESSFULLY
